In [ ]:
#data for lung cancer protein

from Bio import Entrez, SeqIO
import pandas as pd
from io import StringIO
import time
import certifi
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

Entrez.email = "lokanathavighnesh@gmail.com"

# ---------------------------------------
# 1️⃣ SEARCH FOR 1000 mRNA SEQUENCES
# ---------------------------------------

search_handle = Entrez.esearch(
    db="nucleotide",
    term="EGFR AND Homo sapiens AND lung cancer",
    retmax=1000
)

search_results = Entrez.read(search_handle)
search_handle.close()

id_list = search_results["IdList"]

print("Total IDs retrieved:", len(id_list))

# ---------------------------------------
# 2️⃣ FETCH SEQUENCES IN BATCHES
# ---------------------------------------

sequence_data = []

batch_size = 100

for start in range(0, len(id_list), batch_size):
    end = min(start + batch_size, len(id_list))
    batch_ids = id_list[start:end]

    print(f"Fetching records {start} to {end}")

    fetch_handle = Entrez.efetch(
        db="nucleotide",
        id=",".join(batch_ids),
        rettype="fasta",
        retmode="text"
    )

    fasta_data = fetch_handle.read()
    fetch_handle.close()

    records = SeqIO.parse(StringIO(fasta_data), "fasta")

    for record in records:
        sequence_data.append({
            "Accession_ID": record.id,
            "Description": record.description,
            "Sequence": str(record.seq)
        })

    time.sleep(1)  # NCBI rate limit safety

# ---------------------------------------
# 3️⃣ SAVE TO EXCEL
# ---------------------------------------

df = pd.DataFrame(sequence_data)

df.head()


In [ ]:
# data for non cancer protein

from Bio import Entrez, SeqIO
import pandas as pd
from io import StringIO
import time
import ssl

# Temporary SSL bypass (since you had issues)
ssl._create_default_https_context = ssl._create_unverified_context

Entrez.email = "lokanathavighnesh@gmail.com"

# ---------------------------------------
# 1️⃣ SEARCH PROTEIN DATABASE
# ---------------------------------------

search_handle = Entrez.esearch(
    db="protein",
    term="EGFR AND Homo sapiens NOT lung cancer",
    retmax=1000
)

search_results = Entrez.read(search_handle)
search_handle.close()

id_list = search_results["IdList"]

print("Total Protein IDs retrieved:", len(id_list))

# ---------------------------------------
# 2️⃣ FETCH PROTEIN FASTA SEQUENCES
# ---------------------------------------

protein_data = []
batch_size = 100

for start in range(0, len(id_list), batch_size):
    end = min(start + batch_size, len(id_list))
    batch_ids = id_list[start:end]

    print(f"Fetching proteins {start} to {end}")

    fetch_handle = Entrez.efetch(
        db="protein",
        id=",".join(batch_ids),
        rettype="fasta",
        retmode="text"
    )

    fasta_data = fetch_handle.read()
    fetch_handle.close()

    records = SeqIO.parse(StringIO(fasta_data), "fasta")

    for record in records:
        protein_data.append({
            "Accession_ID": record.id,
            "Description": record.description,
            "Protein_Sequence": str(record.seq)
        })

    time.sleep(1)

# ---------------------------------------
# 3️⃣ SAVE TO EXCEL
# ---------------------------------------

df = pd.DataFrame(protein_data)
df.head()


In [ ]:
# code to stitch the cancer and non-cancer sequences

import pandas as pd

df1 = pd.read_excel('lung_cancer_1000_sequences.xlsx')
df2 = pd.read_excel('lung_protein_sequences.xlsx')

merged_df = pd.concat([df1, df2], ignore_index=True)

merged_df.to_excel('combined_file.xlsx', index=False)

merged_df.head()

In [ ]:
# code to get wild sequence

from Bio import Entrez
import pandas as pd
import ssl

ssl._create_default_https_context = ssl._create_unverified_context
Entrez.email = "lokanathavighnesh@gmail.com"

search = Entrez.esearch(
    db="protein",
    term="EGFR[Gene] AND Homo sapiens[Organism] AND srcdb_refseq[PROP]",
    retmax=1
)

result = Entrez.read(search)
search.close()

protein_id = result["IdList"][0]

fetch = Entrez.efetch(
    db="protein",
    id=protein_id,
    rettype="fasta",
    retmode="text"
)

wild_type_sequence = fetch.read()

fetch.close()
print(wild_type_sequence)
df = pd.DataFrame({protein_id : [wild_type_sequence]})

df.to_excel("lung_cancer_wildseq.xlsx",index=False)
df.head()


In [ ]:
# code to compare the combined sequence with the wild sequence using global alignment

from Bio import pairwise2
import pandas as pd

# -----------------------------------
# 1️⃣ LOAD FILES
# -----------------------------------

all_sequences_path = "combined_file.xlsx"
wildtype_path = "lung_cancer_wildseq.xlsx"

all_sequences_df = pd.read_excel(all_sequences_path)
wild_df = pd.read_excel(wildtype_path)

sequence_column = "Sequence"

all_sequences = all_sequences_df[sequence_column].dropna().astype(str)
wild_type_sequence = str(wild_df.iloc[0, 0])

print("Total sequences loaded:", len(all_sequences))

# -----------------------------------
# 2️⃣ GLOBAL ALIGNMENT WITH SCORING
# -----------------------------------

# Scoring scheme

def compute_score_percentage(seq1, seq2):
    alignment = pairwise2.align.globalxx(seq1, seq2)[0]
    
    alignment_score = alignment.score
    
    max_possible_score = len(seq1) * 2
    
    percent_score = (alignment_score / max_possible_score) * 100
    
    return percent_score

# -----------------------------------
# 3️⃣ FILTER USING 80% SCORE
# -----------------------------------

filtered_data = []

for idx, seq in enumerate(all_sequences):
    score_percent = compute_score_percentage(wild_type_sequence, seq)
    
    if score_percent <= 20:
        filtered_data.append({
            "Protein_Sequence": seq,
            "Score_Percentage": score_percent
        })
    
    if idx % 50 == 0:
        print(f"Processed {idx} sequences")

# -----------------------------------
# 4️⃣ SAVE OUTPUT
# -----------------------------------

filtered_df = pd.DataFrame(filtered_data)

output_path = "filtered_sequences.xlsx"
filtered_df.head()